# IT3091 — Group 76 | Final Model Comparison and Evaluation

**Task:** Predict hotel booking cancellations using information available at booking confirmation.

The analysis compares group-aware five-fold cross-validation results, selects a model and classification threshold using **training data only**, and reports one independent evaluation on the held-out test set. Model interpretation and operational limitations follow.

## 1. Compare candidate models

Load the saved validation results from the individual modelling notebooks, including the dummy baseline and the retained configuration for each algorithm.

In [1]:
import pandas as pd
from pathlib import Path

metrics_dir = Path("../../results/metrics")

model_files = {
    "Dummy Baseline": "dummy_classifier_cv.csv",
    "Logistic Regression": "logistic_regression_cv.csv",
    "Decision Tree (Tuned)": "decision_tree_tuned_cv.csv",
    "Random Forest (Tuned)": "random_forest_tuned_cv.csv",
    "Gradient Boosting": "gradient_boosting_cv.csv"
}

metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

comparison = pd.DataFrame({
    name: pd.read_csv(metrics_dir / filename).iloc[-1][metrics]
    for name, filename in model_files.items()
}).T.astype(float)

comparison = comparison.sort_values("pr_auc", ascending=False)

display(comparison.round(4))

,accuracy,precision,recall,f1,roc_auc,pr_auc
Random Forest (Tuned),0.7883,0.8891,0.4881,0.6301,0.8845,0.8420
Gradient Boosting,0.7844,0.8213,0.5331,0.6463,0.8655,0.8177
Decision Tree (Tuned),0.7656,0.6514,0.7907,0.7135,0.8704,0.8177
Logistic Regression,0.7774,0.7541,0.5909,0.6622,0.8421,0.7906
Dummy Baseline,0.6302,0.0000,0.0000,0.0000,0.5000,0.3698


## 2. Load development data

Reconstruct the approved 21-feature input set and reuse the saved group-aware holdout split. The test set is not used for tuning or model selection.

In [2]:
import numpy as np

# Load validated data and existing train/test split
data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

# Use the same approved features as the modelling notebooks
excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X_train = df.drop(columns=excluded).iloc[splits["train_idx"]].copy()
y_train = df["is_canceled"].iloc[splits["train_idx"]].copy()

groups_train = pd.util.hash_pandas_object(X_train, index=False)

print("Training shape:", X_train.shape)
print("Cancellation rate:", f"{y_train.mean():.2%}")
print("Unique feature groups:", groups_train.nunique())

Training shape: (95334, 21)
Cancellation rate: 37.01%
Unique feature groups: 65232


## 3. Random Forest: threshold analysis

Use the previously tuned Random Forest. Five-fold out-of-fold probabilities allow different alert thresholds to be evaluated without predicting the held-out test outcomes.

In [3]:
import sys
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold

repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

# Use the previously selected tuned Random Forest
rf_model = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        class_weight=None,
        random_state=42,
        n_jobs=2
    ))
])

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Tuned Random Forest configured for threshold analysis.")
print("Cross-validation folds:", cv.n_splits)

from sklearn.model_selection import cross_val_predict

# Generate cancellation probabilities using out-of-fold predictions
rf_oof_probs = cross_val_predict(
    rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Out-of-fold predictions generated successfully.")
print("Total predictions:", len(rf_oof_probs))
print("Probability range:",
      f"{rf_oof_probs.min():.4f} - {rf_oof_probs.max():.4f}")

Tuned Random Forest configured for threshold analysis.
Cross-validation folds: 5
Out-of-fold predictions generated successfully.
Total predictions: 95334
Probability range: 0.0209 - 0.9798


In [4]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Compare different cancellation-risk thresholds
thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

results = []

for threshold in thresholds:
    predictions = (rf_oof_probs >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, predictions),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions)
    })

threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))

from sklearn.metrics import confusion_matrix

# Selected threshold from the tested candidates
selected_threshold = 0.40

rf_predictions = (rf_oof_probs >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_train, rf_predictions
).ravel()

print("Random Forest - Threshold 0.40")
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

,Threshold,Precision,Recall,F1
0,0.25,0.5514,0.9437,0.6961
1,0.30,0.5888,0.9130,0.7159
2,0.35,0.6379,0.8571,0.7315
3,0.40,0.7225,0.7452,0.7337
4,0.45,0.8261,0.5858,0.6855
5,0.50,0.8898,0.4884,0.6306
6,0.55,0.9388,0.4198,0.5802
7,0.60,0.9725,0.3729,0.5391


Random Forest - Threshold 0.40
True Negatives: 49956
False Positives: 10097
False Negatives: 8991
True Positives: 26290


## 4. Decision Tree: compare threshold choices

Apply the same development folds and candidate thresholds to the tuned Decision Tree so the decision is not based on comparing a threshold-adjusted forest against a default-threshold tree.

In [5]:
from sklearn.tree import DecisionTreeClassifier

# Previously selected tuned Decision Tree
dt_model = Pipeline([
    ("preprocessor", build_preprocessor(X_train)),
    ("classifier", DecisionTreeClassifier(
        max_depth=16,
        min_samples_leaf=30,
        class_weight="balanced",
        random_state=42
    ))
])

# Generate out-of-fold cancellation probabilities
dt_oof_probs = cross_val_predict(
    dt_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Decision Tree out-of-fold predictions generated.")
print("Total predictions:", len(dt_oof_probs))

# Evaluate Decision Tree at different thresholds
dt_results = []

for threshold in thresholds:
    predictions = (dt_oof_probs >= threshold).astype(int)

    dt_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, predictions),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions)
    })

dt_threshold_results = pd.DataFrame(dt_results)

display(dt_threshold_results.round(4))

Decision Tree out-of-fold predictions generated.
Total predictions: 95334


,Threshold,Precision,Recall,F1
0,0.25,0.5600,0.9252,0.6977
1,0.30,0.5733,0.9083,0.7029
2,0.35,0.5884,0.8914,0.7088
3,0.40,0.6028,0.8689,0.7118
4,0.45,0.6284,0.8291,0.7149
5,0.50,0.6512,0.7897,0.7138
6,0.55,0.7198,0.6859,0.7025
7,0.60,0.7547,0.6353,0.6899


## 5. Select and train the final model

**Selected:** tuned Random Forest, with a classification threshold of **0.40**, fixed before opening the held-out test results. It recorded a pooled out-of-fold F1 of **0.7337**, compared with **0.7149** for the Decision Tree at its best tested threshold of 0.45. The forest also had the higher five-fold PR-AUC (**0.8420** versus **0.8177**). The Decision Tree offered higher recall at its best-F1 threshold; the chosen threshold reflects a balance between cancellation detection and false alerts.

The selected threshold is optimized on the development predictions, so its development F1 may be somewhat optimistic. The independent held-out test provides the final generalization check.

In [6]:
from sklearn.base import clone

# Final model and threshold selected before test evaluation
final_model = clone(rf_model)
final_threshold = 0.40

# Prepare the untouched test dataset
X_test = df.drop(columns=excluded).iloc[splits["test_idx"]].copy()
y_test = df["is_canceled"].iloc[splits["test_idx"]].copy()

# Train the selected model on the complete training set
final_model.fit(X_train, y_train)

print("Final Random Forest trained successfully.")
print("Selected threshold:", final_threshold)
print("Training records:", len(X_train))
print("Test records:", len(X_test))

Final Random Forest trained successfully.
Selected threshold: 0.4
Training records: 95334
Test records: 23876


## 6. Independent held-out test results

Evaluate the **fixed** Random Forest and threshold once on the 23,876 held-out records. Report the test metrics, confusion counts, and save them in a single CSV; do not retune from these results.

In [7]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# Predict cancellation probabilities on the held-out test set
test_probs = final_model.predict_proba(X_test)[:, 1]
test_predictions = (test_probs >= final_threshold).astype(int)

# Calculate final test metrics
test_metrics = {
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Precision": precision_score(y_test, test_predictions),
    "Recall": recall_score(y_test, test_predictions),
    "F1-score": f1_score(y_test, test_predictions),
    "ROC-AUC": roc_auc_score(y_test, test_probs),
    "PR-AUC": average_precision_score(y_test, test_probs)
}

print("Final Random Forest - Held-Out Test Results\n")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

# Final test confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_test, test_predictions
).ravel()

print("Final Random Forest - Test Confusion Matrix\n")
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)
print("\nTotal test bookings:", tn + fp + fn + tp)

# Save final test metrics and confusion matrix
final_results = {
    "Model": "Tuned Random Forest",
    "Threshold": final_threshold,
    **test_metrics,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

output = Path("../../results/metrics/final_random_forest_test_results.csv")

pd.DataFrame([final_results]).to_csv(output, index=False)

print("Final test results saved:", output)

Final Random Forest - Held-Out Test Results

Accuracy: 0.7939
Precision: 0.7137
Recall: 0.7486
F1-score: 0.7307
ROC-AUC: 0.8857
PR-AUC: 0.8434
Final Random Forest - Test Confusion Matrix

True Negatives: 12280
False Positives: 2678
False Negatives: 2242
True Positives: 6676

Total test bookings: 23876
Final test results saved: ..\..\results\metrics\final_random_forest_test_results.csv


Final test results saved: ..\..\results\metrics\final_random_forest_test_results.csv


## 7. Interpret the final model

Inspect Random Forest impurity-based importance for both encoded categories and the original 21 input columns. These scores describe the fitted model, **not causal effects**, and can favor high-cardinality categorical features.

In [8]:
# Extract feature importance from the final Random Forest
feature_names = final_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = final_model.named_steps[
    "classifier"
].feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Display the 15 most important encoded features
display(
    feature_importance
    .sort_values("Importance", ascending=False)
    .head(15)
    .round(4)
)

# Combine encoded feature importance by original column
grouped_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": [
        feature_importance.loc[
            (feature_importance["Feature"] == f"num__{col}") |
            feature_importance["Feature"].str.startswith(f"cat__{col}_"),
            "Importance"
        ].sum()
        for col in X_train.columns
    ]
})

display(
    grouped_importance
    .sort_values("Importance", ascending=False)
    .head(10)
    .round(4)
)

,Feature,Importance
163,cat__country_PRT,0.1753
0,num__lead_time,0.1575
10,num__previous_cancellations,0.0856
210,cat__market_segment_Groups,0.0649
230,cat__customer_type_Transient,0.0474
212,cat__market_segment_Online TA,0.0444
231,cat__customer_type_Transient-Party,0.0416
209,cat__market_segment_Direct,0.0235
211,cat__market_segment_Offline TA/TO,0.0231
217,cat__distribution_channel_TA/TO,0.0231


,Feature,Importance
12,country,0.2500
13,market_segment,0.1615
1,lead_time,0.1575
20,customer_type,0.0941
16,previous_cancellations,0.0856
14,distribution_channel,0.0468
0,hotel,0.0275
3,arrival_date_month,0.0219
7,stays_in_week_nights,0.0213
2,arrival_date_year,0.0201


## 8. Conclusions, recommendations and limitations

### Model Interpretation

The final Random Forest model was interpreted using impurity-based feature importance. The encoded importance scores were aggregated to identify the most influential original booking features.

Country (0.2500), market segment (0.1615), lead time (0.1575), customer type (0.0941), and previous cancellations (0.0856) had the highest importance scores.

These findings suggest that booking characteristics, reservation timing, and historical cancellation behavior contribute to cancellation-risk prediction. However, feature importance does not establish causation or indicate the direction of an effect. The relatively high importance of country also requires caution because categorical cardinality and geographic patterns may affect interpretation and generalization.

### Business Recommendations

The selected model can support hotel revenue management and operations teams by identifying bookings with elevated cancellation risk.

- **Targeted follow-up:** Use high-risk predictions to prioritize appropriate reservation confirmations and customer communication.
- **Operational planning:** Consider aggregated cancellation-risk estimates when planning room availability and staffing.
- **Decision support:** Use predictions alongside operational judgment rather than automatically cancelling reservations or making overbooking decisions.
- **Performance monitoring:** Monitor false cancellation alerts, missed cancellations, and model performance over time.

### Limitations and Responsible AI

The model was developed using historical hotel booking data, so its performance may change when booking patterns or customer behavior change.

The group-aware validation strategy reduces overlap between identical feature combinations, but it does not establish performance on future time periods or different hotels.

Country and customer-related features may reflect dataset-specific patterns. Their use requires fairness checks and monitoring to avoid inappropriate treatment of customers.

Feature importance describes model behavior rather than causal relationships. The financial costs of false positives and false negatives were not quantified, so the selected classification threshold may need business validation before deployment.

### Final Conclusion

The tuned Random Forest was selected using development-data model comparison and threshold analysis. The final classification threshold was fixed at 0.40 before independent test evaluation.

On 23,876 held-out test bookings, the model achieved:

- Accuracy: 0.7939
- Precision: 0.7137
- Recall: 0.7486
- F1-score: 0.7307
- ROC-AUC: 0.8857
- PR-AUC: 0.8434

The model correctly identified 6,676 cancellations, missed 2,242 cancellations, and generated 2,678 false cancellation alerts.

These results demonstrate promising predictive performance for supporting hotel cancellation-risk management. Further evaluation of business costs, fairness, and performance over time would be necessary before operational deployment.